In [62]:
using Plots, ProgressMeter, Statistics, BSON
include("analysis_tools.jl")

In [58]:
nx=100
L=10.0
ν=0.1
k=2
u_mean=1.0
u_amplitude=0.5
noise_strength=0.0005f0
t_end=15
cfl=0.8

X, Δt = burgers_FV(nx, L, ν, k, u_mean, u_amplitude, noise_strength, t_end, cfl);

In [61]:
cfls = 0.4:0.01:1.2
samples=30

X = zeros(Float32, nx, 1, samples*length(cfls))
Δt = zeros(Float32, 1, samples*length(cfls))
y = zeros(Float32, nx, 1, samples*length(cfls))

count=1
@showprogress for cfl in cfls
    X_data, Δt_data = burgers_FV(nx, L, ν, k, u_mean, u_amplitude, noise_strength, t_end, cfl)
    y_data, _ = burgers_FV(nx, L, ν, k, u_mean, u_amplitude, 0, t_end, cfl)

    sample_idx = rand(1:length(X_data)-1, samples)
    for t in sample_idx
        X[:,:,count] .= X_data[t]
        Δt[:,count] .= Δt_data

        y[:,:,count] .= y_data[t+1]
        count+=1
    end
end
Xμ, Xσ = mean(X), std(X)
X = (X .- Xμ) ./ Xσ
y = (y .- Xμ) ./ Xσ;

Progress: 100%|█████████████████████████████████████████| Time: 0:00:09


100×1×2430 Array{Float32, 3}:
[:, :, 1] =
 1.1932956
 1.2760714
 1.3564012
 1.4340719
 1.5088563
 1.5805181
 1.6488084
 1.7134635
 1.774207
 1.8307445
 1.882765
 1.929938
 1.9719142
 ⋮
 0.077451535
 0.17533985
 0.27287003
 0.36988753
 0.46623614
 0.5617585
 0.65629363
 0.7496782
 0.841744
 0.93231905
 1.0212259
 1.1082819

[:, :, 2] =
  0.31401736
  0.34300914
  0.3718899
  0.40064326
  0.42925188
  0.45769393
  0.48594335
  0.51397014
  0.54174
  0.5692106
  0.59633225
  0.6230463
  0.64928246
  ⋮
 -0.03877652
 -0.009193136
  0.020376367
  0.049926553
  0.07945279
  0.10894909
  0.13840893
  0.1678274
  0.19719633
  0.22650868
  0.25575626
  0.28492928

[:, :, 3] =
 -0.9190015
 -0.86314523
 -0.8068312
 -0.750106
 -0.6930126
 -0.63559157
 -0.577881
 -0.519917
 -0.46173373
 -0.40336403
 -0.34483975
 -0.2861914
 -0.22744943
  ⋮
 -1.5281779
 -1.4840302
 -1.4381704
 -1.3908087
 -1.3421209
 -1.2922553
 -1.2413377
 -1.1894773
 -1.136769
 -1.0832969
 -1.0291346
 -0.97434944

;;; … 

[:, :, 24

In [63]:
BSON.@save "data/burgers_dataset.bson" X Δt y Xμ Xσ